# **Gold Layer**

### **Dim Reseller**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# READ DATA FROM SILVER LAYER
silver_reseller = spark.table("accenture_final_project.silver_layer.reseller")

# DEFINE WINDOW SPECIFICATION FOR SURROGATE KEY
# Order by the business key to ensure deterministic sequential numbering
window_spec = Window.orderBy(col("ResellerKey"))

# ADD SURROGATE KEY TO DATAFRAME 
# Named reseller_sk (Surrogate Key) to strictly distinguish it from the Business Key
dim_reseller_df = silver_reseller.withColumn("ResellerSK", row_number().over(window_spec))

# REORDER COLUMNS (SURROGATE KEY MUST BE FIRST) 
# Industry standard: SK is always the first column on the left
dim_reseller_df = dim_reseller_df.select(
    "ResellerSK",     # 1st: Surrogate Key
    "ResellerKey",     # 2nd: Business Key
    "Reseller",
    "BusinessType",
    "City",
    "StateProvince",
    "CountryRegion"
)

# SAVE TO GOLD LAYER AND DISPLAY 
# Save the dimension table using Delta format for performance and reliability
(
    dim_reseller_df.write 
         .format("delta") 
         .mode("overwrite") 
         .saveAsTable("accenture_final_project.gold_layer.dim_reseller")
)

display(dim_reseller_df)